> **Public portfolio copy:** Saved execution outputs have been removed. The notebook source code and explanations are retained. Running the notebook requires a separately authorised DICOM input file in the local `Data/` folder. No Stanford COCA image or dataset file is included in this repository.

# DICOM Image Processing and Segmentation

**Project 2: DICOM Extension – Medical Image Viewer and Preprocessing**

## Overview

This notebook explores fundamental medical image processing and segmentation techniques using a CT image stored in DICOM format.

The analysis uses Hounsfield Units (HU) to identify selected intensity ranges, create binary masks, isolate the patient body region and demonstrate image-processing operations.

## Objectives

- Perform HU-based intensity thresholding.
- Create and visualize binary segmentation masks.
- Separate the patient body from external air.
- Apply edge detection and morphological operations.
- Perform connected-component analysis.
- Extract a Region of Interest (ROI).
- Apply and verify a cleaned patient-body mask.

**Dataset:** One CT DICOM image.

## 1. Environment Setup and DICOM Image Loading

This section imports the required libraries, loads the DICOM image, extracts its pixel data and converts the pixel values into Hounsfield Units (HU).

In [ ]:
# IMPORT REQUIRED LIBRARIES

# Import pathlib for file path management
from pathlib import Path

# Import NumPy for numerical operations
import numpy as np

# Import Matplotlib for image visualization
import matplotlib.pyplot as plt

# Import pydicom for reading DICOM files
import pydicom

# Define project and data directories
project_path = Path.cwd().parent
data_path = project_path / "Data"

# Load the DICOM file
dicom_file = data_path / "IM-0001-0001.dcm"
dcm = pydicom.dcmread(dicom_file)

# Extract the original pixel array
pixel_array = dcm.pixel_array

# Convert pixel values to Hounsfield Units (HU)
ct_image = (
    pixel_array * float(dcm.RescaleSlope)
    + float(dcm.RescaleIntercept)
)

# Display basic image information
print("Image shape:", ct_image.shape)
print("Data type:", ct_image.dtype)
print("Minimum HU:", ct_image.min())
print("Maximum HU:", ct_image.max())

## 2. HU-Based Tissue Segmentation

This section uses selected Hounsfield Unit ranges to identify air, fat-range and soft-tissue-range pixels. Binary masks are created, quantified and visualized, including overlays on the original CT image.

In [ ]:
# IDENTIFY TISSUE REGIONS USING HU THRESHOLDING

# Create binary masks for selected HU ranges
air_mask = ct_image < -500

fat_mask = (ct_image >= -190) & (ct_image <= -30)

soft_tissue_mask = (ct_image > -30) & (ct_image <= 150)

# Count pixels identified in each category
print("Air pixels:", np.sum(air_mask))
print("Fat-range pixels:", np.sum(fat_mask))
print("Soft-tissue-range pixels:", np.sum(soft_tissue_mask))

# Calculate percentage of image occupied by each category
total_pixels = ct_image.size

print("\nPercentage of image:")
print("Air:", np.sum(air_mask) / total_pixels * 100, "%")
print("Fat range:", np.sum(fat_mask) / total_pixels * 100, "%")
print("Soft-tissue range:", np.sum(soft_tissue_mask) / total_pixels * 100, "%")

In [ ]:
# VISUALIZE THE THREE HU-BASED SEGMENTATION MASKS

# Create a figure with three subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Display air segmentation mask
axes[0].imshow(air_mask, cmap="gray")
axes[0].set_title("Air Mask (< -500 HU)")
axes[0].axis("off")

# Display fat segmentation mask
axes[1].imshow(fat_mask, cmap="gray")
axes[1].set_title("Fat-Range Mask (-190 to -30 HU)")
axes[1].axis("off")

# Display soft-tissue-range segmentation mask
axes[2].imshow(soft_tissue_mask, cmap="gray")
axes[2].set_title("Soft-Tissue-Range Mask (-30 to 150 HU)")
axes[2].axis("off")

# Adjust layout and display
plt.tight_layout()
plt.show()

In [ ]:
# OVERLAY SEGMENTATION MASKS ON THE ORIGINAL CT IMAGE

# Create three comparison plots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Define segmentation masks and their titles
masks = [air_mask, fat_mask, soft_tissue_mask]

titles = [
    "Air Mask Overlay",
    "Fat-Range Mask Overlay",
    "Soft-Tissue Mask Overlay"
]

# Display each mask over the original CT image
for ax, mask, title in zip(axes, masks, titles):

    # Display original CT image as the background
    ax.imshow(ct_image, cmap="gray")

    # Overlay selected pixels using a semi-transparent colour
    ax.imshow(
        np.ma.masked_where(~mask, mask),
        cmap="autumn",
        alpha=0.5
    )

    # Set title and remove axes
    ax.set_title(title)
    ax.axis("off")

# Adjust layout and display
plt.tight_layout()
plt.show()

## 3. Patient Body Masking and Internal Air Identification

This section separates the patient body from external air using thresholding, morphological closing and hole filling. The resulting body mask is then used to identify and visualize internal air regions.

In [ ]:
# CREATE A BODY MASK TO SEPARATE THE PATIENT FROM EXTERNAL AIR

# Import image-processing functions
from scipy import ndimage

# Identify pixels likely belonging to the body
body_threshold = ct_image > -500

# Close small gaps in the body boundary
body_closed = ndimage.binary_closing(
    body_threshold,
    structure=np.ones((5, 5))
)

# Fill internal holes to create a continuous body region
body_mask = ndimage.binary_fill_holes(body_closed)

# Display the body mask
plt.figure(figsize=(7, 7))
plt.imshow(body_mask, cmap="gray")
plt.title("Patient Body Mask")
plt.axis("off")
plt.show()

# Display the number of pixels identified as the body region
print("Body mask pixels:", np.sum(body_mask))

In [ ]:
# IDENTIFY INTERNAL AIR USING THE BODY MASK

# Restrict the original air mask to the patient's body region
internal_air_mask = air_mask & body_mask

# Display the internal air mask
plt.figure(figsize=(7, 7))
plt.imshow(internal_air_mask, cmap="gray")
plt.title("Internal Air Mask (< -500 HU)")
plt.axis("off")
plt.show()

# Count selected internal air pixels
print("Internal air pixels:", np.sum(internal_air_mask))

In [ ]:
# OVERLAY THE INTERNAL AIR MASK ON THE ORIGINAL CT IMAGE

# Create the figure
plt.figure(figsize=(8, 8))

# Display the original CT image
plt.imshow(ct_image, cmap="gray")

# Overlay the internal air mask in red
plt.imshow(
    np.ma.masked_where(~internal_air_mask, internal_air_mask),
    cmap="autumn",
    alpha=0.6
)

# Add title and remove axes
plt.title("Internal Air Segmentation Overlay")
plt.axis("off")

# Display the result
plt.show()

## 4. Sobel Edge Detection

This section applies the Sobel operator to demonstrate edge detection in the original CT image.

In [ ]:
# APPLY SOBEL EDGE DETECTION TO THE CT IMAGE

# Import the Sobel edge detection function
from skimage.filters import sobel

# Apply Sobel edge detection
edge_image = sobel(ct_image)

# Display the original and edge-detected images
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Display original CT image
axes[0].imshow(ct_image, cmap="gray")
axes[0].set_title("Original CT Image")
axes[0].axis("off")

# Display detected edges
axes[1].imshow(edge_image, cmap="gray")
axes[1].set_title("Sobel Edge Detection")
axes[1].axis("off")

# Display comparison
plt.tight_layout()
plt.show()

## 5. Morphological Image Processing

This section demonstrates morphological erosion, dilation, opening and closing to examine how these operations modify binary body masks.

In [ ]:
# APPLY MORPHOLOGICAL EROSION AND DILATION TO THE BODY MASK

# Define a 5 × 5 structuring element
structure = np.ones((5, 5))

# Apply erosion to shrink the body mask
eroded_mask = ndimage.binary_erosion(
    body_mask,
    structure=structure
)

# Apply dilation to expand the body mask
dilated_mask = ndimage.binary_dilation(
    body_mask,
    structure=structure
)

# Display the original, eroded and dilated masks
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original body mask
axes[0].imshow(body_mask, cmap="gray")
axes[0].set_title("Original Body Mask")
axes[0].axis("off")

# Eroded body mask
axes[1].imshow(eroded_mask, cmap="gray")
axes[1].set_title("Eroded Body Mask")
axes[1].axis("off")

# Dilated body mask
axes[2].imshow(dilated_mask, cmap="gray")
axes[2].set_title("Dilated Body Mask")
axes[2].axis("off")

# Display comparison
plt.tight_layout()
plt.show()

In [ ]:
# APPLY MORPHOLOGICAL OPENING AND CLOSING TO THE BODY MASK

# Import morphological operations
from scipy.ndimage import binary_opening, binary_closing

# Define a 5 x 5 structuring element
structure = np.ones((5, 5), dtype=bool)

# Apply morphological opening to remove small unwanted regions
opened_mask = binary_opening(body_mask, structure=structure)

# Apply morphological closing to fill small holes and gaps
closed_mask = binary_closing(body_mask, structure=structure)

# Display original, opened and closed masks
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(body_mask, cmap="gray")
axes[0].set_title("Original Body Mask")
axes[0].axis("off")

axes[1].imshow(opened_mask, cmap="gray")
axes[1].set_title("Morphological Opening")
axes[1].axis("off")

axes[2].imshow(closed_mask, cmap="gray")
axes[2].set_title("Morphological Closing")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 6. Connected-Component Analysis and Mask Refinement

This section identifies connected regions in the patient body mask and retains the largest connected component to remove smaller disconnected regions.

In [ ]:
# IDENTIFY CONNECTED REGIONS IN THE PATIENT BODY MASK

# Import connected-component labeling function
from scipy.ndimage import label

# Label all connected foreground regions
labeled_mask, number_of_components = label(body_mask)

# Count the pixels belonging to each component
component_sizes = np.bincount(labeled_mask.ravel())

# Exclude the background (label 0)
component_sizes = component_sizes[1:]

# Display the number and sizes of detected components
print("Number of connected components:", number_of_components)
print("Largest component size:", component_sizes.max())
print("Smallest component size:", component_sizes.min())

In [ ]:
# RETAIN ONLY THE LARGEST CONNECTED COMPONENT

# Identify the largest foreground component, excluding background label 0
largest_component_label = np.argmax(np.bincount(labeled_mask.ravel())[1:]) + 1

# Create a cleaned mask containing only the largest component
cleaned_body_mask = labeled_mask == largest_component_label

# Display the original and cleaned body masks
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(body_mask, cmap="gray")
axes[0].set_title("Original Body Mask")
axes[0].axis("off")

axes[1].imshow(cleaned_body_mask, cmap="gray")
axes[1].set_title("Largest Connected Component")
axes[1].axis("off")

plt.tight_layout()
plt.show()

# Display the number of retained pixels
print("Original body pixels:", np.sum(body_mask))
print("Cleaned body pixels:", np.sum(cleaned_body_mask))

## 7. Region of Interest Extraction and Body Mask Application

This section extracts the patient body region using bounding-box cropping, applies the cleaned body mask to the original CT image and verifies that pixels outside the mask have been set to zero.

In [ ]:
# EXTRACT THE PATIENT BODY REGION USING THE CLEANED MASK

# Find the coordinates of all pixels belonging to the body
rows, cols = np.where(cleaned_body_mask)

# Determine the bounding box of the patient body
row_min, row_max = rows.min(), rows.max()
col_min, col_max = cols.min(), cols.max()

# Crop the original CT image using the bounding box
body_roi = ct_image[
    row_min:row_max + 1,
    col_min:col_max + 1
]

# Display the extracted region
plt.figure(figsize=(7, 7))
plt.imshow(body_roi, cmap="gray")
plt.title("Extracted Patient Body Region (ROI)")
plt.axis("off")
plt.show()

# Display the dimensions of the extracted region
print("Original image shape:", ct_image.shape)
print("Extracted ROI shape:", body_roi.shape)

In [ ]:
# APPLY THE CLEANED BODY MASK TO THE ORIGINAL CT IMAGE

# Retain original HU values inside the body and set outside pixels to zero
masked_ct_image = np.where(cleaned_body_mask, ct_image, 0)

# Display original and masked CT images side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(ct_image, cmap="gray")
axes[0].set_title("Original CT Image")
axes[0].axis("off")

axes[1].imshow(masked_ct_image, cmap="gray")
axes[1].set_title("CT Image After Body Masking")
axes[1].axis("off")

plt.tight_layout()
plt.show()

# Compare image dimensions
print("Original image shape:", ct_image.shape)
print("Masked image shape:", masked_ct_image.shape)

In [ ]:
# VERIFY THAT PIXELS OUTSIDE THE BODY MASK ARE ZERO

# Count pixels outside the body mask
outside_pixels = np.sum(~cleaned_body_mask)

# Check the values outside the mask
outside_values = masked_ct_image[~cleaned_body_mask]

print("Pixels outside body:", outside_pixels)
print("Minimum outside value:", outside_values.min())
print("Maximum outside value:", outside_values.max())
print("All outside pixels are zero:", np.all(outside_values == 0))

## Conclusion

This notebook explored fundamental medical image processing and segmentation techniques using a CT image in Hounsfield Units (HU).

The following operations were implemented:

1. HU-based tissue segmentation using intensity thresholds.
2. Binary mask creation and visualization.
3. Patient-body segmentation.
4. Morphological operations: erosion, dilation, opening and closing.
5. Connected-component analysis.
6. Largest-component extraction to remove smaller disconnected regions.
7. Region of Interest (ROI) extraction through bounding-box cropping.
8. Application and verification of a patient-body mask.

These techniques demonstrate how medical images can be segmented, refined and prepared for subsequent image analysis.

**Note:** The segmentation masks in this notebook are demonstrations based on intensity thresholds. They are not clinically validated anatomical segmentations.